# Chapter 10 — LLM Agents## ReAct: Reasoning + Acting[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)**No GPU required (uses API). Runtime: ~15 minutes.**Builds a ReAct agent that observes, reasons, acts with tools, and iterates — the agent loop from Chapter 10.

In [ ]:
# This notebook uses the Anthropic API or a local model.
# For Colab: set your API key as a Colab secret named ANTHROPIC_API_KEY

import os, json, re
from typing import Optional

# Simple tool definitions
TOOLS = {
    'calculator': lambda expr: str(eval(expr)),
    'word_count': lambda text: str(len(text.split())),
    'reverse':    lambda text: text[::-1],
}

print('Tools defined:', list(TOOLS.keys()))
print('These simulate the tool-use action space from Chapter 10')

## 10.1 The ReAct Loop: Thought → Action → Observation

In [ ]:
def parse_react_output(llm_output: str) -> dict:
    """Parse Thought/Action/Action Input from LLM output."""
    result = {'thought': None, 'action': None, 'action_input': None, 'final_answer': None}
    
    thought_match = re.search(r'Thought:(.*?)(?=Action:|Final Answer:|$)', llm_output, re.DOTALL)
    if thought_match:
        result['thought'] = thought_match.group(1).strip()
    
    action_match = re.search(r'Action:(.*?)(?=Action Input:|$)', llm_output, re.DOTALL)
    if action_match:
        result['action'] = action_match.group(1).strip()
    
    input_match = re.search(r'Action Input:(.*?)(?=Observation:|$)', llm_output, re.DOTALL)
    if input_match:
        result['action_input'] = input_match.group(1).strip()
    
    final_match = re.search(r'Final Answer:(.*?)$', llm_output, re.DOTALL)
    if final_match:
        result['final_answer'] = final_match.group(1).strip()
    
    return result

def execute_tool(action: str, action_input: str) -> str:
    """Execute a tool and return the observation."""
    tool_name = action.strip().lower().replace(' ', '_')
    if tool_name in TOOLS:
        try:
            return TOOLS[tool_name](action_input)
        except Exception as e:
            return f'Error: {e}'
    return f'Unknown tool: {action}. Available: {list(TOOLS.keys())}'

# Simulate a ReAct trajectory WITHOUT calling an API
SIMULATED_TRAJECTORY = [
    {'step': 1, 'thought': 'I need to calculate 17 * 23 first.', 'action': 'calculator', 'action_input': '17 * 23', 'observation': '391'},
    {'step': 2, 'thought': 'Got 391. Now I need to count words in the result string.', 'action': 'word_count', 'action_input': '391', 'observation': '1'},
    {'step': 3, 'final_answer': '17 × 23 = 391, which contains 1 word.'},
]

print('=== Simulated ReAct Trajectory ===')
for step in SIMULATED_TRAJECTORY:
    print(f'\nStep {step["step"]}:')
    if 'thought' in step:
        print(f'  Thought: {step["thought"]}')
        print(f'  Action: {step["action"]}({step["action_input"]})')
        print(f'  Observation: {step["observation"]}')
    if 'final_answer' in step:
        print(f'  Final Answer: {step["final_answer"]}')

## 10.2 Mapping the Agent Loop to MDPEach step maps directly to the RL MDP formalism:

In [ ]:
# Demonstrate the MDP mapping explicitly
print('=== Agent Loop → MDP Mapping ===')
print()
print('State s_t:  The full context window (system prompt + history up to step t)')
print('Action a_t: The agent\'s next generation (Thought + Action + Action Input)')
print('Reward r_t: 0 at each step (sparse — only at task completion)')
print('Transition: s_{t+1} = s_t + a_t + Observation(a_t)')
print('Episode:    From initial task prompt to Final Answer')
print()

# Show concrete example
mapping = {
    's_0': 'System prompt + task: "Calculate 17*23 and count the words"',
    'a_0': 'Thought: I need to multiply. Action: calculator. Input: 17*23',
    'obs_0': 'Observation: 391',
    's_1': 's_0 + a_0 + obs_0  [context grows by one turn]',
    'a_1': 'Thought: Got 391. Action: word_count. Input: 391',
    'obs_1': 'Observation: 1',
    's_2': 's_1 + a_1 + obs_1',
    'a_2': 'Final Answer: 391 contains 1 word',
    'r_T': '+1  [correct answer verified]',
}
for k, v in mapping.items():
    print(f'  {k}: {v}')

## ✅ Key Takeaways1. The ReAct loop (Thought → Action → Observation → Thought...) IS the MDP loop2. Context window = state. Tool call = action. Tool output = observation.3. Reward is typically sparse (only at task completion)4. RL training teaches better action selection at each step → better task completion